In [3]:
import pandas as pd
import os
from shapely.validation import make_valid 
from shapely.geometry import Point, MultiPolygon  
import pathlib
from tqdm import tqdm
import geopandas as gpd 

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [2]:
stations = pd.read_parquet("GEBCO/merged_all_gebco.parquet")

cols = [
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "datasetKey",
    "datasetTitle",
    "sourceDatabase",
    "year",
    "gebcoDepth",
    "startDayOfYear",
    "gear", 
]
all_stations = stations[cols].drop_duplicates()

total_stations = len(all_stations)
print(f"Total unique stations: {total_stations}")

all_stations.to_parquet("Geo/all_stations.parquet", index=False)

all_stations

Total unique stations: 239108


,decimalLatitudeStart,decimalLongitudeStart,datasetKey,datasetTitle,sourceDatabase,year,gebcoDepth,startDayOfYear,gear
0,73.833333,25.891500,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2016,-457.0,274,"Beamtrawl,Subsample method: Sieve content - Me..."
1,68.184360,10.338713,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2012,-875.0,123,"Beamtrawl,Subsample method: Sieve content - Me..."
2,67.035000,10.991333,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2012,-243.0,131,"Beamtrawl,Subsample method: Sieve content - Me..."
3,70.602667,20.689000,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2010,-118.0,264,"Beamtrawl,Subsample method: Sieve content - Me..."
4,70.245000,31.055000,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2014,-167.0,106,"Beamtrawl,Subsample method: Sieve content - Me..."
...,...,...,...,...,...,...,...,...,...
2365212,68.906255,14.356099,37e512e6-732f-49fb-a4bd-544522b42b25,Lophelia pertusa reefs in Norwegian seawaters,OBIS,<NA>,-238.0,<NA>,<NA>
2365223,68.931869,14.381949,e5389590-6d7e-4752-b84b-faa2a411db87,ICES Vulnerable Marine Ecosystems,OBIS,<NA>,-245.0,<NA>,<NA>
2365224,68.071999,39.521999,a145f369-b272-46f7-86bc-08412b1ee927,Arctic benthic invertebrate collection of the ...,OBIS,1876,2.0,193,<NA>
2365232,73.500000,131.673004,a145f369-b272-46f7-86bc-08412b1ee927,Arctic benthic invertebrate collection of the ...,OBIS,1993,-21.0,232,<NA>


In [4]:
# helper: drop rings < 4 points and invalid geometries  
def clean_poly(g):  
    if g is None: return None  
    try: g = make_valid(g)  
    except Exception: return None  
    if g.is_empty or not g.is_valid: return None  
    if g.geom_type == "Polygon":  
        return g if len(g.exterior.coords) >= 4 else None  
    if g.geom_type == "MultiPolygon":  
        parts = [p for p in g.geoms if len(p.exterior.coords) >= 4]  
        if not parts: return None  
        return MultiPolygon(parts) if len(parts) > 1 else parts[0]  
    return None  
  
# 1 ▸ device points  
df  = pd.read_parquet("Geo/all_stations.parquet")  
pts = gpd.GeoDataFrame(  
    df,  
    geometry=[Point(xy) for xy in zip(df["decimalLongitudeStart"], df["decimalLatitudeStart"])],  
    crs="EPSG:4326",  
)  
print("Station rows:", len(pts))  
  
# 2 ▸ shapefiles in Geo/  
shp_dir = pathlib.Path("Geo")  
shps    = sorted(shp_dir.glob("*.shp"))  
print("Found", len(shps), "shapefiles in Geo/")  
  
joined = pts.copy()  
  
for shp in shps:  
    name = shp.stem  
    print("•", name)  
    raw = gpd.read_file(shp)  
    prop_cols = [c for c in raw.columns if c.lower() in ["class", "geomorphic"]]  
  
    # clean layer  
    records = []  
    for _, row in tqdm(raw.iterrows(), total=len(raw), leave=False):  
        geom = clean_poly(row.geometry)  
        if geom is None: continue  
        rec  = {name + "_" + c: row[c] for c in prop_cols}  
        rec["geometry"] = geom  
        records.append(rec)  
  
    if not records:  
        print("  ↳ skipped (no valid polygons)"); continue  
  
    layer = gpd.GeoDataFrame(records, crs=raw.crs).to_crs("EPSG:4326")  
    joined = gpd.sjoin(joined, layer, how="left", predicate="intersects").drop(columns="index_right")  

# Handle duplicate columns by renaming with suffix - after all joins complete
print("Checking for duplicate columns...")
cols = joined.columns.tolist()
new_cols = []
seen = {}
for col in cols:
    if col in seen:
        seen[col] += 1
        new_cols.append(f"{col}_{seen[col]}")
        print(f"  Renamed duplicate '{col}' → '{col}_{seen[col]}'")
    else:
        seen[col] = 0
        new_cols.append(col)
joined.columns = new_cols
  
# 3 ▸ reorder columns  
order = [  
    "Hadal", "Abyss", "Slope", "Shelf", "Abyssal_Classification", "Shelf_Classification",  
    "Plateaus", "Terraces", "Trenches", "Rises", "Fans", "Spreading_ridges", "Ridges",  
    "Troughs", "Rift_valleys", "Shelf_valleys", "Glacial_troughs", "Sills",  
    "Bridges", "Seamounts", "Guyots", "Canyons", "Escarpments", "Basins"  
]  
  
# build full ordered list: point columns first, then *_Class/_Geomorphic in order  
base_cols   = [c for c in joined.columns if c not in joined.geometry.name and not any(c.startswith(x + "_") for x in order)]  
ordered_cols = base_cols.copy()  
added = set(base_cols)  # Track which columns we've already added
for feat in order:  
    for c in joined.columns:
        if c.startswith(feat + "_") and c not in added:
            ordered_cols.append(c)
            added.add(c)
  
joined = joined[ordered_cols + [joined.geometry.name]]  
  
# 4 ▸ save + preview  
out_parquet = "Geo/all_stations.parquet"  
joined.drop(columns="geometry").to_parquet(out_parquet, index=False)  
print("✓ saved to", out_parquet)  
print(joined.drop(columns='geometry').head())

Station rows: 239108
Found 24 shapefiles in Geo/
• Abyss


• Abyssal_Classification


• Basins


• Bridges


• Canyons


• Escarpments


... (16 outputs left out)

• Trenches


• Troughs


Checking for duplicate columns...


✓ saved to Geo/all_stations.parquet
   decimalLatitudeStart  decimalLongitudeStart  \
0             73.833333              25.891500   
1             68.184360              10.338713   
2             67.035000              10.991333   
3             70.602667              20.689000   
4             70.245000              31.055000   

                             datasetKey           datasetTitle sourceDatabase  \
0  99006141-bf02-4821-8e6e-5818a1cbf70a  imr_mareano_beamtrawl           GBIF   
1  99006141-bf02-4821-8e6e-5818a1cbf70a  imr_mareano_beamtrawl           GBIF   
2  99006141-bf02-4821-8e6e-5818a1cbf70a  imr_mareano_beamtrawl           GBIF   
3  99006141-bf02-4821-8e6e-5818a1cbf70a  imr_mareano_beamtrawl           GBIF   
4  99006141-bf02-4821-8e6e-5818a1cbf70a  imr_mareano_beamtrawl           GBIF   

   year  gebcoDepth  startDayOfYear  \
0  2016      -457.0             274   
1  2012      -875.0             123   
2  2012      -243.0             131   
3  2010      -118.0 

In [5]:
input_file = "Geo/all_stations.parquet"   
df_geo = pd.read_parquet(input_file)  
  
print("Rows:", len(df_geo))  

# 2. Create the target column and initialize with empty strings  
df_geo["geomorphologicalCategory"] = ""  

# 3.1  Shelf + high  ➜  Deep shelf  
mask = (  
    df_geo["Shelf_Geomorphic"].str.contains("Shelf", case=False, na=False)  
    & df_geo["Shelf_Classification_Class"].str.contains("high", case=False, na=False)  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Deep shelf"  
  
# 3.2  Shelf + medium  ➜  Medium shelf  
mask = (  
    df_geo["Shelf_Geomorphic"].str.contains("Shelf", case=False, na=False)  
    & df_geo["Shelf_Classification_Class"].str.contains("medium", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Medium shelf"  
  
# 3.3  Shelf + low  ➜  Shallow shelf  
mask = (  
    df_geo["Shelf_Geomorphic"].str.contains("Shelf", case=False, na=False)  
    & df_geo["Shelf_Classification_Class"].str.contains("low", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Shallow shelf"  
  
# 3.4  Slope  ➜  Slope  
mask = (  
    df_geo["Slope_Geomorphic"].str.contains("Slope", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Slope"  
  
# 3.5  Ridges_Geomorphic with “Ridge”  ➜  Ridges  
mask = (  
    df_geo["Ridges_Geomorphic"].str.contains("Ridge", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Ridges"  
  
# 3.6  Spreading_ridges_Geomorphic with “Spreading Ridge”  ➜  Ridges  
mask = (  
    df_geo["Spreading_ridges_Geomorphic"].str.contains("Spreading Ridge", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Ridges"  
  
# 3.7  Rift_valleys_Geomorphic with “Rift Valley”  ➜  Ridges  
mask = (  
    df_geo["Rift_valleys_Geomorphic"].str.contains("Rift Valley", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Ridges"  
  
# 3.8  Abyssal_Classification_Class with “Hills” or “Mountains”  ➜  Abyssal mountains and hills
mask = (  
    df_geo["Abyssal_Classification_Class"].str.contains("Hills|Mountains", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal mountains and hills"  
  
# 3.9  Seamounts_Geomorphic with “Seamount”  ➜  Abyssal mountains and hills 
mask = (  
    df_geo["Seamounts_Geomorphic"].str.contains("Seamount", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal mountains and hills"  
  
# 3.10  Abyssal_Classification_Class with “Plains”  ➜  Abyssal plains and hadal zones
mask = (  
    df_geo["Abyssal_Classification_Class"].str.contains("Plains", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal plains and hadal zones"  
  
# 3.11  Hadal_Geomorphic with “Hadal”  ➜  Abyssal plains and hadal zones  
mask = (  
    df_geo["Hadal_Geomorphic"].str.contains("Hadal", case=False, na=False)  
    & (df_geo["geomorphologicalCategory"] == "")  
)  
df_geo.loc[mask, "geomorphologicalCategory"] = "Abyssal plains and hadal zones"  

# Save as excel
output_file = "Geo/all_stations.parquet"  
df_geo.to_parquet(output_file, index=False)  
  
print("Saved new file:", output_file)  

Rows: 239108
Saved new file: Geo/all_stations.parquet


In [7]:
# Delete unneccessary columns
# Load existing collapsed file 
input_file = 'Geo/all_stations.parquet'
df_geo = pd.read_parquet(input_file)

# Retain only desired columns (remove sourceDatabase to avoid duplicates)
keep_cols = ['decimalLatitudeStart', 'decimalLongitudeStart', 'geomorphologicalCategory']
df_geo = df_geo[keep_cols]

# Deduplicate on (lat, lon, year) - same location should have one geomorphologicalCategory
df_geo = df_geo.drop_duplicates(subset=['decimalLatitudeStart', 'decimalLongitudeStart'], keep='first')

# Save to a new file
output_file = 'Geo/all_stations.parquet'
df_geo.to_parquet(output_file, index=False)

# Show head for confirmation
print(df_geo.head())
print('Saved file as', output_file)
print(f'Total unique station-years: {len(df_geo)}')

   decimalLatitudeStart  decimalLongitudeStart geomorphologicalCategory
0             73.833333              25.891500             Medium shelf
1             68.184360              10.338713                    Slope
2             67.035000              10.991333             Medium shelf
3             70.602667              20.689000               Deep shelf
4             70.245000              31.055000               Deep shelf
Saved file as Geo/all_stations.parquet
Total unique station-years: 195652


In [8]:
# Calculate nr of rows that are lacking a geo category

df = pd.read_parquet('Geo/all_stations.parquet')

missing_mask = df['geomorphologicalCategory'].isna() | (df['geomorphologicalCategory'].str.strip() == '')
missing_count = missing_mask.sum()
print('Missing rows count:', missing_count)

Missing rows count: 13745


In [10]:
# Load the two parquet files
all_merged = pd.read_parquet("GEBCO/merged_all_gebco.parquet")
geo_collapsed = pd.read_parquet("Geo/all_stations.parquet")

# Merge on the combination of decimalLatitudeStart, decimalLongitudeStart
all_merged_geo = all_merged.merge(
    geo_collapsed[['decimalLatitudeStart', 'decimalLongitudeStart', 'geomorphologicalCategory']],
    on=['decimalLatitudeStart', 'decimalLongitudeStart'],
    how='left'
)

# Save the merged dataframe
all_merged_geo.to_parquet("Geo/merged_all_geo.parquet", index=False)

print(f"Merged dataframe shape: {all_merged_geo.shape}")
print(f"Original all_merged: {len(all_merged)} records")
print(f"geomorphologicalCategory column added with {all_merged_geo['geomorphologicalCategory'].notna().sum()} non-null values")
print(all_merged_geo[['decimalLatitudeStart', 'decimalLongitudeStart', 'geomorphologicalCategory']].head(10))

Merged dataframe shape: (2365237, 70)
Original all_merged: 2365237 records
geomorphologicalCategory column added with 2365237 non-null values
   decimalLatitudeStart  decimalLongitudeStart geomorphologicalCategory
0             73.833333              25.891500             Medium shelf
1             68.184360              10.338713                    Slope
2             67.035000              10.991333             Medium shelf
3             70.602667              20.689000               Deep shelf
4             70.245000              31.055000               Deep shelf
5             70.619667              18.175333             Medium shelf
6             64.978157               6.894050             Medium shelf
7             68.481000              10.930167                    Slope
8             63.885040               7.568300             Medium shelf
9             75.710200              18.987730             Medium shelf


In [11]:
# Count records per geomorphologicalCategory in all_merged_geo.parquet
all_merged_geo = pd.read_parquet("Geo/merged_all_geo.parquet")

geo_category_counts = all_merged_geo['geomorphologicalCategory'].value_counts(dropna=False)

print("Records per geomorphologicalCategory:")
print(geo_category_counts)
print(f"\nTotal records: {len(all_merged_geo)}")

Records per geomorphologicalCategory:
geomorphologicalCategory
Deep shelf                        1058783
Medium shelf                       802607
Shallow shelf                      183873
Slope                              172640
                                   100104
Abyssal mountains and hills         23311
Ridges                              15050
Abyssal plains and hadal zones       8869
Name: count, dtype: int64

Total records: 2365237
